# 50 — Kernel quantistico: geometria (E6), rilevamento one-class e costi (E7)

**Feature map della specifica:**
- `RY(x_j)` su ogni qubit, poi un anello di CNOT, poi re-uploading `RY(0,7·x_j)`;
- kernel di fedeltà K_Q(x,x') = |⟨ψ(x)|ψ(x')⟩|²;
- controllo **separabile**: stesse rotazioni, senza CNOT.

Gli stati sono simulati esattamente con numpy (`qdetect.qkernel`), verificati contro PennyLane nei test automatici.

**Preprocessing** (stimato solo sulle righe benigne di training), come nel notebook del relatore: standardizzazione, PCA a q componenti, angoli π·tanh(z/2).

1. **Geometria (E6):** geometric difference g_CQ rispetto a kernel classici (lineare, RBF, polinomiali, separabile) con ridge dichiarato, rango effettivo e concentrazione fuori diagonale. È un *prescreening*, non una prova di vantaggio.
2. **One-class:** One-Class SVM con kernel precalcolato (quantistico, separabile, RBF, polinomiale), addestrato su 1.500 righe benigne, con soglia conforme e metriche a rischio fissato come nel notebook 40. Va oltre la specifica, che si ferma alla geometria.
3. **Costi (E7):** shot necessari (stima di Hoeffding) e tempi di simulazione. Nessun tempo è un tempo su QPU.

In [1]:
import sys, time
from pathlib import Path
REPO = Path.cwd()
while not (REPO / 'qdetect').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.metrics.pairwise import rbf_kernel, polynomial_kernel, linear_kernel
import qdetect as qd
from qdetect import oneclass as O, qkernel as QK
pd.set_option('display.width', 220)
SEED = 42
split = qd.load_split(REPO / 'splits' / 'split_oneclass_v1.npz')
names = split.features
tr, cal, te, to = split.idx_train_large, split.idx_val, split.idx_test, split.extra['test_original']
y = split.y
f_idx = names.index('freq1')

def angles(feature_set, q):
    cols = [names.index(c) for c in O.FEATURE_SETS[feature_set]]
    ben = tr[y[tr] == 0]
    s1 = StandardScaler().fit(split.X[np.ix_(ben, cols)])
    pca = PCA(n_components=min(q, len(cols)), random_state=SEED).fit(s1.transform(split.X[np.ix_(ben, cols)]))
    s2 = StandardScaler().fit(pca.transform(s1.transform(split.X[np.ix_(ben, cols)])))
    f = lambda idx: np.pi * np.tanh(s2.transform(pca.transform(s1.transform(split.X[np.ix_(idx, cols)]))) / 2)
    return f, pca.explained_variance_ratio_.sum()

CONFIGS = {'all7, 6 qubit': ('all7', 6), 'core5, 5 qubit': ('core5', 5)}

# Calibrazione nel dominio dei vettori nuovi (vedi notebook 40, sezione 8): benigni del
# pool del test nuovo non usati nel test e con vettori diversi da quelli del test.
g_all = qd.feature_groups(split.X)
used = np.zeros(len(y), bool); used[te] = True; used[to] = True
cand = np.flatnonzero((split.source == 1) & ~used & (y == 0) & ~np.isin(g_all, np.unique(g_all[te])))
cal_new = np.random.default_rng(7).choice(cand, min(20_000, len(cand)), replace=False)
CALS = {'cal. file di training': cal[y[cal] == 0], 'cal. dominio nuovo': cal_new}

## 1. Geometria (E6): 5 estrazioni di 96 righe di training

In [2]:
geo = []
for cname, (fs, q) in CONFIGS.items():
    f, evr = angles(fs, q)
    for draw in range(5):
        ids = np.random.default_rng(SEED + draw).choice(tr, 96, replace=False)
        x = f(ids); N = len(x)
        Kq = QK.fidelity_kernel(x)
        controls = {'linear': linear_kernel(x), 'separable (quantum, no CNOT)': QK.fidelity_kernel(x, entangled=False)}
        for g in [0.1, 1 / q, 1.0]:
            controls[f'rbf γ={g:.3g}'] = rbf_kernel(x, gamma=g)
        for d in [2, 3, 4]:
            controls[f'poly d={d}'] = polynomial_kernel(x, degree=d, gamma=1 / q, coef0=1)
        base = dict(config=cname, q=q, pca_var=evr, draw=draw, N=N, eff_rank_q=QK.effective_rank(Kq), **QK.offdiag_stats(Kq))
        for name, K in controls.items():
            for ridge in [1e-6, 1e-4, 1e-2, 0.1]:
                geo.append(dict(base, control=name, ridge=ridge, g=QK.geometric_difference(Kq, K, ridge)))
geo = pd.DataFrame(geo); geo.to_csv(REPO / 'results' / 'qkernel_geometry.csv', index=False)
summary = geo.groupby(['config', 'control', 'ridge']).g.agg(['mean', 'std', 'min', 'max']).round(3)
display(summary)
display(geo.groupby('config')[['eff_rank_q', 'offdiag_mean', 'offdiag_std', 'pca_var']].mean().round(3))

mean      std       min       max
config         control                      ridge                                          
all7, 6 qubit  linear                       0.000001  2194.447  129.563  2012.279  2332.333
                                            0.000100   219.447   12.956   201.231   233.236
                                            0.010000    21.971    1.295    20.150    23.349
                                            0.100000     7.023    0.407     6.448     7.456
               poly d=2                     0.000001  1512.434   49.709  1463.016  1575.748
...                                                        ...      ...       ...       ...
core5, 5 qubit rbf γ=1                      0.100000     1.914    0.050     1.854     1.966
               separable (quantum, no CNOT) 0.000001   446.358  131.223   321.288   669.092
                                            0.000100    99.397    9.904    91.045   115.749
                                            0.010000    12.887    1.490    10.511    14.583
                                            0.100000     4.524    0.433     3.978     5.153

[64 rows x 4 columns]

,eff_rank_q,offdiag_mean,offdiag_std,pca_var
config,,,,
"all7, 6 qubit",34.391,0.047,0.129,1.0
"core5, 5 qubit",27.671,0.069,0.146,1.0


**Come leggerlo.** g ≈ 1 significa che il kernel classico "contiene" quello quantistico: nessun margine, secondo il criterio di Huang et al. Valori grandi dipendono molto dal ridge: un g che cresce solo con ridge piccolissimo riflette il malcondizionamento della matrice, non una geometria utile. Il criterio originale riguarda modelli supervisionati, non il rilevamento one-class.

## 2. One-Class SVM con kernel precalcolato, a rischio fissato

In [3]:
N_TRAIN = 1500
rows, timing = [], []
for cname, (fs, q) in CONFIGS.items():
    f, _ = angles(fs, q)
    ben = np.random.default_rng(SEED).choice(tr[y[tr] == 0], N_TRAIN, replace=False)
    Xb = f(ben); Xcals = {k: f(v) for k, v in CALS.items()}; Xte = f(te); Xto = f(to)
    kernels = {
        'quantum (CNOT ring)': lambda A, B: QK.fidelity_kernel(A, B, entangled=True),
        'quantum separable': lambda A, B: QK.fidelity_kernel(A, B, entangled=False),
        'rbf γ=1/q': lambda A, B: rbf_kernel(A, B, gamma=1 / q),
        'poly d=3': lambda A, B: polynomial_kernel(A, B, degree=3, gamma=1 / q, coef0=1),
    }
    for kname, kf in kernels.items():
        t0 = time.perf_counter(); Ktr = kf(Xb, Xb); oc = OneClassSVM(kernel='precomputed', nu=0.05).fit(Ktr); t_fit = time.perf_counter() - t0
        t0 = time.perf_counter()
        sc = lambda X: -oc.decision_function(kf(X, Xb)).ravel()
        s_cals = {k: sc(v) for k, v in Xcals.items()}; s_te, s_to = sc(Xte), sc(Xto); t_inf = time.perf_counter() - t0
        timing.append(dict(config=cname, kernel=kname, fit_s=t_fit, score_s=t_inf, n_scored=sum(map(len, Xcals.values())) + len(Xte) + len(Xto)))
        for cal_name, s_cal in s_cals.items():
            for tname, s, yy, st in [('test nuovo', s_te, y[te], split.X[te, f_idx]), ('test originale', s_to, y[to], split.X[to, f_idx])]:
                if cal_name == 'cal. dominio nuovo' and tname == 'test originale':
                    continue
                for r in O.risk_metrics(s, yy, s_cal, st):
                    rows.append(dict(config=cname, kernel=kname, calibration=cal_name, test=tname, **r))
        last = [r for r in rows if r['config'] == cname and r['kernel'] == kname and r['stratum'] == 'all' and r['alpha'] == 0.01 and r['calibration'] == 'cal. dominio nuovo']
        print(f'{cname:15s} {kname:20s} ' + '  '.join(f"{r['test']}: AUROC={r['auroc']:.3f} TPR@1%={r['tpr']:.3f}" for r in last), flush=True)
qoc = pd.DataFrame(rows); qoc.to_csv(REPO / 'results' / 'qkernel_oneclass.csv', index=False)
T = qoc[qoc.stratum == 'all']
display(T.pivot_table(index=['config', 'kernel'], columns=['calibration', 'test', 'alpha'], values='tpr').round(3))
display(T.pivot_table(index=['config', 'kernel'], columns=['calibration', 'test', 'alpha'], values='fpr').round(4))
display(T[T.alpha == 0.01].pivot_table(index=['config', 'kernel'], columns='test', values='auroc').round(4))

all7, 6 qubit   quantum (CNOT ring)  test nuovo: AUROC=0.629 TPR@1%=0.071


all7, 6 qubit   quantum separable    test nuovo: AUROC=0.626 TPR@1%=0.018


all7, 6 qubit   rbf γ=1/q            test nuovo: AUROC=0.738 TPR@1%=0.057


all7, 6 qubit   poly d=3             test nuovo: AUROC=0.181 TPR@1%=0.000


core5, 5 qubit  quantum (CNOT ring)  test nuovo: AUROC=0.671 TPR@1%=0.045


core5, 5 qubit  quantum separable    test nuovo: AUROC=0.488 TPR@1%=0.010


core5, 5 qubit  rbf γ=1/q            test nuovo: AUROC=0.731 TPR@1%=0.058


core5, 5 qubit  poly d=3             test nuovo: AUROC=0.224 TPR@1%=0.000


calibration                        cal. dominio nuovo               cal. file di training                                           
test                                       test nuovo                          test nuovo               test originale              
alpha                                          0.0001 0.0010 0.0100                0.0001 0.0010 0.0100         0.0001 0.0010 0.0100
config         kernel                                                                                                               
all7, 6 qubit  poly d=3                           0.0  0.000  0.000                   0.0  0.000  0.000            0.0  0.000  0.000
               quantum (CNOT ring)                0.0  0.019  0.071                   0.0  0.086  0.278            0.0  0.199  0.288
               quantum separable                  0.0  0.000  0.018                   0.0  0.007  0.126            0.0  0.002  0.169
               rbf γ=1/q                          0.0  0.013  0.057                   0.0  0.083  0.543            0.0  0.181  0.570
core5, 5 qubit poly d=3                           0.0  0.000  0.000                   0.0  0.000  0.000            0.0  0.000  0.000
               quantum (CNOT ring)                0.0  0.011  0.045                   0.0  0.086  0.286            0.0  0.113  0.323
               quantum separable                  0.0  0.001  0.010                   0.0  0.001  0.025            0.0  0.000  0.016
               rbf γ=1/q                          0.0  0.008  0.058                   0.0  0.087  0.501            0.0  0.134  0.546

calibration                        cal. dominio nuovo                 cal. file di training                                               
test                                       test nuovo                            test nuovo                 test originale                
alpha                                          0.0001  0.0010  0.0100                0.0001  0.0010  0.0100         0.0001  0.0010  0.0100
config         kernel                                                                                                                     
all7, 6 qubit  poly d=3                        0.0001  0.0017  0.0109                   0.0  0.0019  0.0059            0.0  0.0005  0.0077
               quantum (CNOT ring)             0.0001  0.0008  0.0107                   0.0  0.0174  0.1464            0.0  0.0008  0.0090
               quantum separable               0.0001  0.0009  0.0103                   0.0  0.0054  0.0534            0.0  0.0009  0.0107
               rbf γ=1/q                       0.0001  0.0013  0.0111                   0.0  0.0183  0.2359            0.0  0.0008  0.0101
core5, 5 qubit poly d=3                        0.0000  0.0011  0.0109                   0.0  0.0001  0.0017            0.0  0.0003  0.0078
               quantum (CNOT ring)             0.0001  0.0011  0.0110                   0.0  0.0223  0.1351            0.0  0.0008  0.0103
               quantum separable               0.0001  0.0011  0.0107                   0.0  0.0012  0.0214            0.0  0.0008  0.0094
               rbf γ=1/q                       0.0000  0.0010  0.0102                   0.0  0.0158  0.2344            0.0  0.0007  0.0101

test                                test nuovo  test originale
config         kernel                                         
all7, 6 qubit  poly d=3                 0.1814          0.0127
               quantum (CNOT ring)      0.6287          0.8468
               quantum separable        0.6257          0.8102
               rbf γ=1/q                0.7378          0.9672
core5, 5 qubit poly d=3                 0.2242          0.0204
               quantum (CNOT ring)      0.6714          0.9232
               quantum separable        0.4877          0.4327
               rbf γ=1/q                0.7314          0.9660

### Test di DeLong: kernel quantistico contro i controlli (test nuovo)

In [4]:
dl = []
for cname, (fs, q) in CONFIGS.items():
    f, _ = angles(fs, q)
    ben = np.random.default_rng(SEED).choice(tr[y[tr] == 0], N_TRAIN, replace=False)
    Xb, Xte = f(ben), f(te)
    sc = {}
    for kname, kf in {'quantum (CNOT ring)': lambda A, B: QK.fidelity_kernel(A, B),
                      'quantum separable': lambda A, B: QK.fidelity_kernel(A, B, entangled=False),
                      'rbf γ=1/q': lambda A, B: rbf_kernel(A, B, gamma=1 / q)}.items():
        oc = OneClassSVM(kernel='precomputed', nu=0.05).fit(kf(Xb, Xb))
        sc[kname] = -oc.decision_function(kf(Xte, Xb)).ravel()
    for other in ['quantum separable', 'rbf γ=1/q']:
        dl.append(dict(config=cname, confronto=f'quantum (CNOT ring) vs {other}', **qd.delong_test(y[te], sc['quantum (CNOT ring)'], sc[other])))
pd.DataFrame(dl).round(4)

,config,confronto,auroc_a,auroc_b,diff,z,p_value
0,"all7, 6 qubit",quantum (CNOT ring) vs quantum separable,0.6287,0.6257,0.0030,0.9747,0.3297
1,"all7, 6 qubit",quantum (CNOT ring) vs rbf γ=1/q,0.6287,0.7378,-0.1091,-54.8018,0.0000
2,"core5, 5 qubit",quantum (CNOT ring) vs quantum separable,0.6714,0.4877,0.1837,55.6575,0.0000
3,"core5, 5 qubit",quantum (CNOT ring) vs rbf γ=1/q,0.6714,0.7314,-0.0600,-33.2687,0.0000


## 3. Costi (E7)

In [5]:
cost = []
for N in [96, 1500]:
    pairs = N * (N - 1) // 2
    for eps in [0.05, 0.02, 0.01]:
        spp = QK.hoeffding_shots(pairs, eps)
        cost.append(dict(fase=f'Gram di training N={N}', overlaps=pairs, epsilon=eps, shots_per_overlap=spp, total_shots=pairs * spp))
for eps in [0.05, 0.02, 0.01]:
    n_ov = N_TRAIN  # una riga di test richiede un overlap con ogni riga di training
    spp = QK.hoeffding_shots(n_ov, eps)
    cost.append(dict(fase='una predizione (contro 1.500 righe)', overlaps=n_ov, epsilon=eps, shots_per_overlap=spp, total_shots=n_ov * spp))
cost = pd.DataFrame(cost); cost.to_csv(REPO / 'results' / 'qkernel_cost.csv', index=False)
display(cost)
display(pd.DataFrame(timing).round(3))
print('Una QPU a ~1e4 shot/s impiegherebbe circa',
      round(cost[(cost.fase.str.startswith('una')) & (cost.epsilon == 0.05)].total_shots.iloc[0] / 1e4), 's per UNA predizione (ε=0,05).')

,fase,overlaps,epsilon,shots_per_overlap,total_shots
0,Gram di training N=96,4560,0.05,2423,11048880
1,Gram di training N=96,4560,0.02,15143,69052080
2,Gram di training N=96,4560,0.01,60570,276199200
3,Gram di training N=1500,1124250,0.05,3525,3962981250
4,Gram di training N=1500,1124250,0.02,22027,24763854750
5,Gram di training N=1500,1124250,0.01,88108,99055419000
6,una predizione (contro 1.500 righe),1500,0.05,2201,3301500
7,una predizione (contro 1.500 righe),1500,0.02,13753,20629500
8,una predizione (contro 1.500 righe),1500,0.01,55011,82516500


,config,kernel,fit_s,score_s,n_scored
0,"all7, 6 qubit",quantum (CNOT ring),0.016,2.099,149903
1,"all7, 6 qubit",quantum separable,0.022,2.033,149903
2,"all7, 6 qubit",rbf γ=1/q,0.019,1.392,149903
3,"all7, 6 qubit",poly d=3,0.032,2.157,149903
4,"core5, 5 qubit",quantum (CNOT ring),0.012,0.980,149903
5,"core5, 5 qubit",quantum separable,0.030,0.959,149903
6,"core5, 5 qubit",rbf γ=1/q,0.015,1.124,149903
7,"core5, 5 qubit",poly d=3,0.022,2.172,149903


Una QPU a ~1e4 shot/s impiegherebbe circa 330 s per UNA predizione (ε=0,05).
